In [1]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [2]:
organism = "homo_sapiens"
cell_source = "bone"
cell_state = None

reference = "GRCh38"
paper_doi = "https://doi.org/10.1038/s41422-021-00467-z"
table_link = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41422-021-00467-z/MediaObjects/41422_2021_467_MOESM7_ESM.xlsx"

# don't include in header
table_name = "41422_2021_467_MOESM7_ESM.xlsx"
table_name = "clean_degs.xlsx" 

## Read in file

In [3]:
excel = pd.read_excel(table_name, sheet_name = ["Degs human_limb&bone Fig. 1d", "Degs human long bone Fig. S3b", "Degs human calvaria Fig. 6b"], skiprows = 0)

df = pd.concat(excel.values(), ignore_index = True)

In [4]:
df.head()

,cluster,gene,logfoldchanges,pvals_adj
0,LBM1,H2AFZ,1.734192,0.0
1,LBM1,RANBP1,1.468134,0.0
2,LBM1,NaN,1.090531,0.0
3,LBM1,DUSP6,2.428796,0.0
4,LBM1,HMGA1,1.814476,0.0


## Unfiltered

In [5]:
unfiltered_df = df.copy()
unfiltered_df.rename(columns ={"cluster": "cell_type_label"}, inplace=True)
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["gene_id"] = None

unfiltered_df.drop(['logfoldchanges', 'pvals_adj'], axis=1, inplace=True) 
save = unfiltered_df.to_dict(orient = "records")

## Save unfiltered

In [6]:
with open('evidence_unfiltered.json', 'w') as f:
    json.dump(save,f, indent = 4)

## Perform the filtering

In [5]:
col_pval = "pvals_adj"
col_lfc = "logfoldchanges"
col_gene = "gene"
col_ct = "cluster" # says cluster in table, but before this had "celltype"
col_rank = None # set to None if pct. 1DNE

In [6]:
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]

if col_rank:
    m = m.sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [7]:
markers

,cluster,gene,logfoldchanges,pvals_adj
8,LBM1,PMAIP1,2.701771,0.000000e+00
23,LBM1,VEGFD,2.299420,0.000000e+00
38,LBM1,LHX2,3.078587,0.000000e+00
64,LBM1,MYCN,2.345730,0.000000e+00
91,LBM1,MSX1,2.377080,0.000000e+00
...,...,...,...,...
4123,Endothelium,SULT1C4,3.322689,9.311667e-24
4124,Endothelium,TSPAN15,3.671395,1.090203e-23
4126,Endothelium,THSD1,4.319571,4.554218e-23
4128,Endothelium,SMTN,2.418430,5.822234e-24


## Find Ensembl ID


In [8]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [9]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=5):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [10]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json


In [11]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["gene_id"] = df["gene"]
df["gene_id"] = df["gene_id"].apply(run)
save = df.to_dict(orient = "records")

In [12]:
save

[{'cell_type_label': 'LBM1',
  'gene': 'PMAIP1',
  'organism': 'homo_sapiens',
  'cell_source': 'bone',
  'cell_state': None,
  'gene_id': 'ENSG00000141682'},
 {'cell_type_label': 'LBM1',
  'gene': 'VEGFD',
  'organism': 'homo_sapiens',
  'cell_source': 'bone',
  'cell_state': None,
  'gene_id': 'ENSG00000165197'},
 {'cell_type_label': 'LBM1',
  'gene': 'LHX2',
  'organism': 'homo_sapiens',
  'cell_source': 'bone',
  'cell_state': None,
  'gene_id': 'ENSG00000106689'},
 {'cell_type_label': 'LBM1',
  'gene': 'MYCN',
  'organism': 'homo_sapiens',
  'cell_source': 'bone',
  'cell_state': None,
  'gene_id': 'ENSG00000134323'},
 {'cell_type_label': 'LBM1',
  'gene': 'MSX1',
  'organism': 'homo_sapiens',
  'cell_source': 'bone',
  'cell_state': None,
  'gene_id': 'ENSG00000163132'},
 {'cell_type_label': 'LBM3',
  'gene': 'EMX2',
  'organism': 'homo_sapiens',
  'cell_source': 'bone',
  'cell_state': None,
  'gene_id': 'ENSG00000170370'},
 {'cell_type_label': 'LBM3',
  'gene': 'NR2F1',
  'orga

## Save evidence

In [13]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 